In [1]:
import feedparser
import pandas as pd
from datetime import date, timedelta
from sqlalchemy import create_engine

engine = create_engine("sqlite:///c:\\ruby\\portlt\\db\\development.sqlite3")
conlt = engine.connect()

clean_path = "../data/clean/"
raw_path = "../data/raw/"
csv_path = "\\Users\\User\\iCloudDrive\\"
box_path = "\\Users\\User\\Dropbox\\"
one_path = "\\Users\\User\\OneDrive\\Documents\\Data\\"
pdf_path = "../PDF/"

pd.set_option('display.max_colwidth', 255)
pd.set_option('display.max_rows',None)
url = "https://feeds.feedburner.com/Setorth-form45-en"

today = date.today()
year = 2022
mmdd_str = today.strftime('%m%d')
mmdd_str

'0208'

In [2]:
today = date(2023, 2, 7)
mmdd_str = today.strftime('%m%d')
mmdd_str

'0207'

In [3]:
rss_source = feedparser.parse(url)
f45_number = len(rss_source.entries)
print("Number of F45: ", f45_number)

Number of F45:  8


In [4]:
f45_items = []

for x in range(f45_number):
    f45_content = rss_source.entries[x]
    f45_item = {}
    
    print("\n----------------------------------\n")
    
    print("F45: " + str(x))
    title_ary = f45_content.title.partition(' ')
    f45_item['name'] = title_ary[0].strip() 
    print("Title: ", f45_item['name'])  
    f45_item['year'] = year
    print("Year: ", f45_item['year'])      
    qtr_ary = title_ary[2].partition(' (F45)')
    f45_item['quarter'] = qtr_ary[0][-1]    
    print("Quarter: ", f45_item['quarter'])    
    f45_item['link'] = f45_content.link
    print("Link: ", f45_item['link'])
    f45_item['published'] = f45_content.published
    print("Published: ", f45_item['published'])  
    f45_items.append(f45_item)


----------------------------------

F45: 0
Title:  ENGY
Year:  2022
Quarter:  e
Link:  https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16757269707760&symbol=ENGY
Published:  Tue, 07 Feb 2023 19:25:30 +0700

----------------------------------

F45: 1
Title:  FPT
Year:  2022
Quarter:  1
Link:  https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16757269706080&symbol=FPT
Published:  Tue, 07 Feb 2023 18:29:05 +0700

----------------------------------

F45: 2
Title:  LUXF
Year:  2022
Quarter:  2
Link:  https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16757269705640&symbol=LUXF
Published:  Tue, 07 Feb 2023 18:19:51 +0700

----------------------------------

F45: 3
Title:  KCE
Year:  2022
Quarter:  y
Link:  https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16757269703790&symbol=KCE
Published:  Tue, 07 Feb 2023 17:41:16 +0700

----------------------------------

F45: 4
Title:  ENGY
Year:  2022
Quarter:  e
Link:  https://www.set.or.th/en/market/

In [5]:
df = pd.DataFrame(f45_items)
df[['name','year','quarter','published']]

,name,year,quarter,published
0,ENGY,2022,e,"Tue, 07 Feb 2023 19:25:30 +0700"
1,FPT,2022,1,"Tue, 07 Feb 2023 18:29:05 +0700"
2,LUXF,2022,2,"Tue, 07 Feb 2023 18:19:51 +0700"
3,KCE,2022,y,"Tue, 07 Feb 2023 17:41:16 +0700"
4,ENGY,2022,e,"Tue, 07 Feb 2023 17:35:43 +0700"
5,IRPC,2022,y,"Tue, 07 Feb 2023 17:18:24 +0700"
6,SVR,2022,3,"Tue, 07 Feb 2023 09:00:49 +0700"
7,SVR,2022,y,"Tue, 07 Feb 2023 08:33:49 +0700"


In [9]:
df.groupby(['year','quarter']).count()

name  link  published
year quarter                       
2022 1           1     1          1
     2           1     1          1
     3           1     1          1
     4           5     5          5

In [10]:
df.loc[(df.quarter == '2') ,['year','quarter']] = ['2022','4']
df.groupby(['year','quarter']).count()

name  link  published
year quarter                       
2022 1           1     1          1
     3           1     1          1
     4           6     6          6

In [7]:
df.loc[(df.quarter == 'e') ,['year','quarter']] = ['2022','4']
df.groupby(['year','quarter']).count()

name  link  published
year quarter                       
2022 1           1     1          1
     2           1     1          1
     3           1     1          1
     y           3     3          3
     4           2     2          2

In [8]:
df.loc[(df.quarter == 'y') ,['year','quarter']] = ['2022','4']
df.groupby(['year','quarter']).count()

name  link  published
year quarter                       
2022 1           1     1          1
     2           1     1          1
     3           1     1          1
     4           5     5          5

In [11]:
df.loc[(df.name == 'FPT') ,['year','quarter']] = ['2023','1']
df.groupby(['year','quarter']).count()

name  link  published
year quarter                       
2022 3           1     1          1
     4           6     6          6
2023 1           1     1          1

### After change all illogical quarters

In [12]:
df.quarter = df.quarter.astype(int)
df.groupby(['year','quarter']).count()

name  link  published
year quarter                       
2022 3           1     1          1
     4           6     6          6
2023 1           1     1          1

In [13]:
### First equals to latest published pdf file
df = df.drop_duplicates(subset='name', keep='first')
df.shape

(6, 5)

In [14]:
file_name = 'F45-RAW-' + mmdd_str + '.csv'
raw_file = raw_path + file_name
output_file = csv_path + file_name
box_file = box_path + file_name
one_file = one_path + file_name

df[['name','year','quarter','published']].to_csv(output_file, header=True, index=False, sep=',')
df[['name','year','quarter','published']].to_csv(box_file,    header=True, index=False, sep=',')
df[['name','year','quarter','published']].to_csv(one_file,    header=True, index=False, sep=',')
df[['name','year','quarter','published']].to_csv(raw_file,    header=True, index=False, sep=',')

In [15]:
sql = '''
SELECT *
FROM exempts
ORDER BY name'''
df_exempts = pd.read_sql(sql, conlt)
df_exempts.shape[0]

403

In [16]:
df_merge = pd.merge(df, df_exempts, on='name', how='outer', indicator=True)
df_merge.shape

(409, 7)

### Tickers that are in Exempts table

In [17]:
in_exempts = df_merge.loc[
    df_merge['_merge'] == 'both',
    ['name','year','quarter','published','link']
    
]
in_exempts.year = in_exempts.year.astype(int)
in_exempts.quarter = in_exempts.quarter.astype(int)
in_exempts.sort_values(by=['published'],ascending=[False])

,name,year,quarter,published,link


In [18]:
in_exempts.sort_values(by=['published'],ascending=[False]).shape[0]

0

### Not in exempts table

In [19]:
df_out = df_merge.loc[
    df_merge['_merge'] == 'left_only',
    ['name','year','quarter','published','link']
]
df_out.year = df_out.year.astype(int)
df_out.quarter = df_out.quarter.astype(int)
df_out.sort_values(by=['published'],ascending=[False])

,name,year,quarter,published,link
0,ENGY,2022,4,"Tue, 07 Feb 2023 19:25:30 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16757269707760&symbol=ENGY
1,FPT,2023,1,"Tue, 07 Feb 2023 18:29:05 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16757269706080&symbol=FPT
2,LUXF,2022,4,"Tue, 07 Feb 2023 18:19:51 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16757269705640&symbol=LUXF
3,KCE,2022,4,"Tue, 07 Feb 2023 17:41:16 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16757269703790&symbol=KCE
4,IRPC,2022,4,"Tue, 07 Feb 2023 17:18:24 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16757269701080&symbol=IRPC
5,SVR,2022,3,"Tue, 07 Feb 2023 09:00:49 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16757269689820&symbol=SVR


In [20]:
#df_out = df_out.drop(df_out.index[df_out['name'] == "SCC"])
df_out.shape[0]

6

In [21]:
sql = '''
SELECT *
FROM tickers
ORDER BY name'''
df_tickers = pd.read_sql(sql, conlt)
df_tickers.shape

(401, 9)

In [22]:
df_merge2 = pd.merge(df_out, df_tickers, on='name', how='outer', indicator=True)
df_merge2.shape

(404, 14)

### There are no ticker records

In [23]:
df_merge2.loc[
    df_merge2['_merge'] == 'left_only',
    ['name','year','quarter','published','link']
]

,name,year,quarter,published,link
0,ENGY,2022.0,4.0,"Tue, 07 Feb 2023 19:25:30 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16757269707760&symbol=ENGY
2,LUXF,2022.0,4.0,"Tue, 07 Feb 2023 18:19:51 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16757269705640&symbol=LUXF
5,SVR,2022.0,3.0,"Tue, 07 Feb 2023 09:00:49 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16757269689820&symbol=SVR


In [24]:
df_merge2.loc[
    df_merge2['_merge'] == 'left_only',
    ['name','year','quarter','published','link','id','market']
].shape

(3, 7)

### There are ticker records

In [25]:
df_out2 = df_merge2.loc[
    df_merge2['_merge'] == 'both',
    ['name','year','quarter','published','link','id','market']
]
df_out2

,name,year,quarter,published,link,id,market
1,FPT,2023.0,1.0,"Tue, 07 Feb 2023 18:29:05 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16757269706080&symbol=FPT,746.0,SET
3,KCE,2022.0,4.0,"Tue, 07 Feb 2023 17:41:16 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16757269703790&symbol=KCE,249.0,SET50
4,IRPC,2022.0,4.0,"Tue, 07 Feb 2023 17:18:24 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16757269701080&symbol=IRPC,227.0,SET50 / SETCLMV / SETTHSI


In [31]:
df_out2 = df_out2[df_out2.year.notnull()]
df_out2.shape

(3, 7)

In [32]:
df_out2['year'] = df_out2['year'].astype(int)
df_out2['quarter'] = df_out2['quarter'].astype(int)
df_out2.shape

(3, 7)

In [33]:
file_name = 'F45-CLEAN-' + mmdd_str + '.csv'
clean_file = clean_path + file_name
output_file = csv_path + file_name
box_file = box_path + file_name
one_file = one_path + file_name

df_out2[['name','year','quarter','published','market']].sort_values(['published'],ascending=[False]).to_csv(output_file, header=True, index=False, sep=',')
df_out2[['name','year','quarter','published','market']].sort_values(['published'],ascending=[False]).to_csv(clean_file, header=True, index=False, sep=',')
df_out2[['name','year','quarter','published','market']].sort_values(['published'],ascending=[False]).to_csv(box_file, header=True, index=False, sep=',')
df_out2[['name','year','quarter','published','market']].sort_values(['published'],ascending=[False]).to_csv(one_file, header=True, index=False, sep=',')

In [34]:
sql = '''
SELECT * 
FROM epss
WHERE year = 2022'''
df_epss = pd.read_sql(sql, conlt)
df_epss.shape

(705, 14)

In [35]:
df_merge3 = pd.merge(df_out2, df_epss, on=['name','year','quarter'], how='outer', indicator=True)
df_merge3.shape

(708, 19)

### Already input, display profit amt & eps to check with new F45

In [36]:
df_merge3[df_merge3['_merge'] == 'both']

,name,year,quarter,published,link,id_x,market,id_y,q_amt,y_amt,aq_amt,ay_amt,q_eps,y_eps,aq_eps,ay_eps,ticker_id,publish_date,_merge


In [37]:
df_merge3[df_merge3['_merge'] == 'both'].shape

(0, 19)

### New F-45

In [38]:
df_inp2 = df_merge3[df_merge3['_merge'] == 'left_only']
df_inp3 = df_inp2.copy()
df_inp3.shape

(3, 19)

In [39]:
df_inp3['year'] = df_inp3['year'].astype(str)
df_inp3['quarter'] = df_inp3['quarter'].astype(str)
final = df_inp3.sort_values('name',ascending=True)
final_str = final.name+' '+final.year+' '+final.quarter+' '+final.link
final_str

0      FPT 2023 1 https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16757269706080&symbol=FPT
2    IRPC 2022 4 https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16757269701080&symbol=IRPC
1      KCE 2022 4 https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16757269703790&symbol=KCE
dtype: object

In [40]:
df_inp3_q3 = df_inp3[df_inp3['quarter'] == '4']
final_q3 = df_inp3_q3.sort_values('name',ascending=True)
final_str_q3 = final_q3.name+' '+final_q3.year+' '+final_q3.quarter+' '+final_q3.link
final_str_q3

2    IRPC 2022 4 https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16757269701080&symbol=IRPC
1      KCE 2022 4 https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16757269703790&symbol=KCE
dtype: object

In [41]:
df_inp3_q1 = df_inp3[df_inp3['quarter'] != '4']
final_q1 = df_inp3_q1.sort_values('name',ascending=True)
final_str_q1 = final_q1.name+' '+final_q1.year+' '+final_q1.quarter+' '+final_q1.link
final_str_q1

0    FPT 2023 1 https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16757269706080&symbol=FPT
dtype: object